# Deploying NVIDIA Nemotron 3.5 Lightning with vLLM

This notebook will walk you through how to run the NVIDIA Nemotron 3.5 Lightning NVFP4 checkpoint with vLLM on a single H100.

[vLLM](https://docs.vllm.ai) is a fast and easy-to-use library for LLM inference and serving.

Nemotron 3.5 Lightning is published as two checkpoints:

- **BF16**: [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16)
- **NVFP4**: [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4)

**This notebook runs our recommended configuration: the NVFP4 checkpoint on a single H100 with DSpark speculative decoding.** Commands for the other tested configurations are at the end, under **Additional configurations**.

**Model size:** 30B total parameters, 3B active (MoE)

Prerequisites for this notebook:
- 1x NVIDIA H100 80GB with recent drivers
- [Docker](https://docs.docker.com/engine/install/) with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html)
- Python 3.10+

## Overview

- **Serve** the Nemotron 3.5 Lightning NVFP4 checkpoint on a single H100 using vLLM
- **Query the model** through an OpenAI-compatible API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget
- **Reference commands** for BF16, GB200, DGX Spark, and the other speculative decoding methods

## Table of Contents

1. **Decoding options for this model** - Base, MTP, DFlash, and DSpark
2. **Environment setup** - Container image, client dependencies, and GPU check
   - Launch on NVIDIA Brev
   - Pull the vLLM Docker image
   - Install notebook client dependencies
   - Verify GPU
3. **OpenAI-compatible server** - Launch vLLM and confirm it is ready
   - Launch the Docker container
   - Configuration reference
   - Start server
   - Wait for the server to be ready
4. **Generate responses** - Chat completions, reasoning, and tool calling
   - Client setup
   - Single completion
   - Sequential completions
   - Streamed generation
   - Reasoning
   - Tool calling
   - Controlling reasoning budget
5. **Cleanup and shutdown** - Free the GPU and reset the kernel
6. **Additional configurations** - Reference commands for other hardware and precisions
   - 1x H100
   - 1x GB200
   - 1x DGX Spark


## Decoding options for this model

Nemotron 3.5 Lightning can produce tokens four ways. All four serve the same weights and differ only in how many tokens come out of a single forward pass.

| Option | Where drafts come from | Extra weights to download |
|---|---|---|
| **Base** (no speculative decoding) | nothing, one token per pass | none |
| **MTP** | a prediction layer inside the checkpoint | none |
| **DFlash** | a separate block-diffusion draft model, a whole block per pass | DFlash checkpoint |
| **DSpark** | a separate semi-autoregressive draft model, a whole block per pass | DSpark checkpoint |

The three speculative options share one mechanism: a cheap draft proposes several tokens ahead, the model verifies them all in a single pass, and every token up to the first mismatch is kept. A rejected token invalidates itself and everything after it, so the payoff depends on how often drafts are right - a bad guess costs compute without producing output. Pick exactly one: a server has a single draft path, so the flags conflict at launch rather than stacking.

**Concurrency decides whether speculation pays off.** With few requests in flight the GPU has spare capacity, and speculation spends it to shorten the critical path, which lowers per-request latency. Under heavy load the GPU is already busy producing real tokens, so verifying drafts that end up rejected takes throughput away from queued work. `--max-num-seqs` caps how many requests run at once, so benchmark the base configuration before assuming a speculative one wins under load.

**Draft length is the main knob.** `--speculative_config.num_speculative_tokens` sets how far ahead to guess, at a value validated per configuration below. Raising it improves the best case per step but lowers the odds that the whole run is accepted, and wastes more compute when it is not - lowering it shrinks the win but makes it more consistent.

**Two caches share the GPU.** Attention layers use a KV cache that grows with sequence length, while the Mamba layers keep a fixed-size recurrent state per sequence, halved by `--mamba-ssm-cache-dtype float16`. Both scale with `--max-model-len`, so lowering it frees memory for concurrency. DFlash and DSpark also hold a second set of weights, which is the other reason their practical concurrency ceiling is lower.

This notebook uses **DSpark**.

## Environment setup

### Launch on NVIDIA Brev

You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (1x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Haw6pwtZwgoiPRoO2R4cbiAASY)

### Pull the vLLM Docker image

The model runs inside a vLLM container. Pull it once before starting the server:

```shell
docker pull vllm/vllm-openai:v0.27.1
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `jinja2` renders the chat template it applies.

In [1]:
# Bootstrap pip only if the kernel environment is missing it
import importlib.util, subprocess, sys

if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

%pip install -q openai==2.38.0 transformers==5.9.0 "jinja2>=3.1.0"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Verify GPU

Confirm your GPU is visible on the host before starting the Docker container.

> **Expected output:** One row per GPU, showing an H100 with roughly 80 GB of memory alongside the host driver version. If `nvidia-smi` is not found, the NVIDIA driver is not installed.

In [2]:
# Confirm the GPU is visible on the host
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

index, name, memory.total [MiB], driver_version
0, NVIDIA H100 PCIe, 81559 MiB, 570.148.08


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using vLLM.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the vLLM container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it --gpus all --ipc=host --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  --entrypoint /bin/bash \
  vllm/vllm-openai:v0.27.1
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

Run the `vllm serve` command below from inside this container. The commands in **Additional configurations** at the end run here too.

### Configuration reference

Settings for the configuration this notebook runs.

| Setting | NVFP4 | Why |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` | 4-bit weights fit a 30B MoE on one GPU |
| **Draft model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark` | Proposes the token blocks the model verifies |
| **Served model name** | `nemotron-3.5-lightning` | What the client cells send |
| **Hardware (this notebook)** | 1x H100 80GB | What these values were tuned on |
| **Docker image** | `vllm/vllm-openai:v0.27.1` | First release carrying the quantized DSpark fix |
| **Quantization** | `modelopt_mixed` (auto-detected from the checkpoint) | No flag needed |
| **Mamba backend** | `flashinfer` | Fused kernels for the recurrent layers |
| **Mamba SSM cache** | FP16, stochastic rounding, 5 Philox rounds | Halves state memory |
| **Mamba cache mode** | `align` | Layout this configuration was validated with |
| **Max model length** | 1048576 (1M tokens) | Full context window |
| **Max concurrent sequences** | 128 | Leaves memory for the draft model |
| **Prefix caching** | Enabled | Reuses KV for prompts that share a prefix |
| **Async scheduling** | Enabled | Overlaps scheduling with GPU work |
| **Speculative decoding** | DSpark, 3 draft tokens | Lower latency at this concurrency |
| **Reasoning parser** | `nemotron_v3` | Splits thinking from the final answer |
| **Tool parser** | `qwen3_coder`, with auto tool choice | Turns tool syntax into OpenAI `tool_calls` |
| **Host / port** | `127.0.0.1:8000` | Local-only, matches the client cells |

### Start server

Run the command below from inside the Docker container terminal. This is the configuration the rest of the notebook is written against: the NVFP4 checkpoint on one H100 with DSpark speculative decoding, tuned for interactive use at a concurrency of 128 or below.

> **Note:** The first launch takes a while. The progress readout can sit at `0%` for several minutes at a time: first while the weights download from Hugging Face and load, then while the server compiles CUDA kernels. That is expected rather than a hang, so give it time instead of restarting. Later launches reuse the cached weights and compiled kernels and start much faster.

> **Note:** Parser names are backend-specific: vLLM's `nemotron_v3` is `nemotron-v3` in TensorRT-LLM and `nemotron_3` in SGLang.

> **Note:** `--speculative_config.num_speculative_tokens` must be at least the draft checkpoint's `dspark_block_size`. vLLM refuses to start below that, because a shorter block produces garbled output rather than just fewer accepted tokens.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --max-num-seqs 128 \
  --max-model-len 1048576 \
  --enable-prefix-caching \
  --async-scheduling \
  --speculative_config.method dspark \
  --speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
  --speculative_config.num_speculative_tokens 3 \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --mamba-cache-mode align \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --host 127.0.0.1 \
  --port 8000
```

### Wait for the server to be ready

In a new terminal, poll `/v1/models`:

```shell
until curl -sf http://localhost:8000/v1/models | grep -q nemotron-3.5-lightning; do
  echo "Waiting for server..."; sleep 5
done
echo "Server is ready"
```

Then check what the server is actually serving, and send one short request to confirm it generates:

```shell
curl -s http://localhost:8000/v1/models | python3 -m json.tool

curl -sf http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "nemotron-3.5-lightning", "prompt": "Hello", "max_tokens": 16}'
```

> **Expected output:** The loop prints `Waiting for server...` while the model loads, then `Server is ready` once `nemotron-3.5-lightning` is listed by `/v1/models`. The model list shows that name as the `id` - vLLM also reports the checkpoint path in `root` and the running context length in `max_model_len`. The final command returns a short JSON completion, confirming the model is loaded and generating.

## Generate responses

The cells below show single, sequential, and streamed completions, followed by reasoning on/off, tool calling, and reasoning budget examples.

> **Note:** Reasoning tokens count toward `max_tokens`. If `content` comes back empty or `None`, the reasoning trace consumed the entire budget before the model produced an answer, so raise `max_tokens`.

### Client setup

In [3]:
from openai import OpenAI

# Set this to match the --served-model-name used when starting the server
SERVED_MODEL_NAME = "nemotron-3.5-lightning"
BASE_URL = "http://127.0.0.1:8000/v1"

client = OpenAI(base_url=BASE_URL, api_key="null")

### Single completion

In [8]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Briefly explain: what is vLLM and why is it useful for large model inference?"},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
)
choice = response.choices[0]
print("Reasoning:", choice.message.reasoning)
print("Content:", choice.message.content)

Reasoning: Here's a thinking process:

1.  **Analyze User Request:**
   - User asks: "Briefly explain: what is vLLM and why is it useful for large model inference?"
   - Key requirements: Brief explanation, cover what vLLM is, and why it's useful for large model inference.

2.  **Identify Core Concepts:**
   - vLLM: A Python library/framework for efficient serving of large language models.
   - Key problem it solves: Memory efficiency, throughput, latency for LLM inference.
   - Key technical innovation: PagedAttention (analogous to OS paging for memory management).
   - Benefits: Higher throughput, lower cost, supports longer sequences, better GPU utilization.

3.  **Structure the Answer:**
   - What is vLLM? (concise definition)
   - Why is it useful? (key advantages/features)
   - Keep it brief as requested.

4.  **Draft - Mental Refinement:**
   vLLM is an open-source library designed for high-throughput serving of large language models. Its core innovation is a memory management t

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [10]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=1024,
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12**.

Since \(12 \times 12 = 144\), the principal square root is 12. (Note that \(-12\) is also a square root of 144, as \((-12) \times (-12) = 144\), but the positive value is typically given when asking for "the square root.")

Q: What is the capital of France?
A: The capital of France is Paris.

Q: Explain quantum computing in simple terms.
A: ### The Quick Version
Classical computers use **bits**: tiny switches that are either `0` or `1`. All your photos, apps, and searches are just long strings of these on/off switches.

**Quantum computers use qubits** (quantum bits). Thanks to physics, a qubit can be `0`, `1`, or **both at the same time**—until you measure it. This is called **superposition**.

### Two Key Ideas
1. **Superposition**: Imagine a spinning coin. While it's flipping, it's neither heads nor tails exclusively—it's kind of both. A qubit works like that. Only when you "look" (measure it) does it settle in

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [6]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=True,
)

section = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

Reasoning: Here's a thinking process:

1.  **Analyze User Input**: User asks "What are the first 5 prime numbers?"
2.  **Identify Core Concept**: Prime numbers are natural numbers greater than 1 that have no positive divisors other than 1 and themselves.
3.  **Determine First 5 Primes**: 
   - Start from 2: 2 (prime)
   - 3 (prime)
   - 4 (not prime, divisible by 2)
   - 5 (prime)
   - 6 (not prime)
   - 7 (prime)
   So the first 5 are: 2, 3, 5, 7, 11.
4.  **Formulate Response**: List them clearly, maybe with a brief definition. Keep it concise.
5.  **Self-Check**: Does it answer exactly? Yes. Are they correct? Yes. Format is good.✅

Content: The first 5 prime numbers are: 

**2, 3, 5, 7, 11**

A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself. Note that 2 is the only even prime number.

### Reasoning

The model supports two modes: **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`. Use `temperature=1.0, top_p=0.95` in both modes.

In [13]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
)
print("Reasoning:", resp.choices[0].message.reasoning)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about vLLM."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print("Content:", resp2.choices[0].message.content)

Reasoning on
Reasoning: Here's a thinking process:

1.  Analyze the Request:
   - User wants a haiku about GPUs
   - A haiku is a traditional Japanese poem with a 5-7-5 syllable structure
   - Topic: GPUs (Graphics Processing Units)

2.  Understand the constraints:
   - 3 lines
   - Syllable counts: 5, 7, 5
   - Topic: GPUs

3.  Brainstorm GPU-related concepts:
   - Graphics processing
   - Parallel cores
   - Rendering images
   - Computing power
   - Chips, frames, games, screens, algorithms, deep learning, parallel processing

4.  Draft lines with syllable counts:

   Let's try to fit GPU themes into 5-7-5.

   Option 1:
   Silicon chip (5) - wait, "Sil-i-con chip" is 4 syllables. Let's count carefully.
   "Silicon" = 3 (si-li-con)
   "Chip" = 1
   Total 4. Need 5.

   Let's count syllables properly:
   - "Graphics" = 2 (graph-ics)
   - "Processing" = 3 (pro-cess-ing)
   - "Unit" = 2 (u-nit) wait, "Unit" is 2? Actually "unit" is 2 syllables: u-nit. But sometimes pronounced 1. Let's 

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`, then run the function and hand its result back so the model can answer the user.

> **Note:** When tool calling with reasoning enabled, pass `"force_nonempty_content": true` inside `chat_template_kwargs`. Without it, `content` can come back empty and the server may not surface the reasoning trace and the tool call together - coding agents in particular expect text alongside the call.

In [14]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Calculate the tip amount for a bill",
        "parameters": {
            "type": "object",
            "properties": {
                "bill_total": {"type": "integer", "description": "The total amount of the bill"},
                "tip_percentage": {"type": "integer", "description": "The percentage of tip to apply"},
            },
            "required": ["bill_total", "tip_percentage"],
        },
    },
}]

messages = [{"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages,
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": True, "force_nonempty_content": True}},
)

choice = completion.choices[0]
print("Reasoning:", choice.message.reasoning)
print("Tool calls:", choice.message.tool_calls)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - Bill total: $50
   - Tip percentage: 15%
   - Question: What will be the amount for 15% tip?

2.  **Identify Required Tool:**
   - The `calculate_tip` function takes `bill_total` and `tip_percentage` as integers.
   - `bill_total` = 50
   - `tip_percentage` = 15

3.  **Check Parameter Types & Constraints:**
   - Both are integers, match the schema.
   - No missing required parameters.

4.  **Prepare Function Call:**
   - `calculate_tip(bill_total=50, tip_percentage=15)`

5.  **Execute Function Call:** (Mental simulation or actual call)
   - I'll just output the function call as requested.

   Wait, let me verify the function schema:
   ```json
   {
     "description": "Calculate the tip amount for a bill",
     "name": "calculate_tip",
     "parameters": {
       "properties": {
         "bill_total": {"description": "The total amount of the bill", "type": "integer"},
         "tip_percentage": {"description": "The

In [15]:
import json

def calculate_tip(bill_total, tip_percentage):
    """Return the tip and the final total for a bill."""
    tip = round(bill_total * tip_percentage / 100, 2)
    return {"tip": tip, "total": round(bill_total + tip, 2)}

# Map schema names to real functions, then run whichever one the model picked
tool_functions = {"calculate_tip": calculate_tip}

call = choice.message.tool_calls[0]
result = tool_functions[call.function.name](**json.loads(call.function.arguments))

# Hand the result back so the model can answer the user
followup = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages + [
        choice.message,
        {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)},
    ],
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Tool result:", result)
print("Final answer:", followup.choices[0].message.content)

Tool result: {'tip': 7.5, 'total': 57.5}
Final answer: The 15% tip on a $50 bill is **$7.50**, making the total amount **$57.50**.


### Controlling reasoning budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [ ]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"  # use actual HF model ID for tokenizer
)

In [19]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: Here's a thinking process:

1.  **Analyze the Request:**
   - User wants a haiku about GPUs.
   - A haiku is a Japanese poetic form with a 5-7-5 syllable structure (total 17 syllables).
   - Topic: GPUs (Graphics Processing Units).

2.  **Understand the Syllable Structure:**
   - Line 1: 5 syllables
   - Line 2: 7 syllables
   - Line 3: 5 syllables

3.  **Brainstorm GPU-related concepts:**
   - Graphics, processing, cores, pixels, frames,.
Content: Processing pixels swift,
accelerating every frame drawn,
speed in silicon minds.


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the Docker container in the terminal where it was started (`Ctrl+C`). The `--rm` flag automatically removes the container on exit.
2. Restart the kernel if needed to ensure a clean state.

## Additional configurations

Reference commands for the configurations this notebook does not run.

Each entry gives a full base command followed by the flags that switch on a speculator. Add a flag block to the base command - the blocks are fragments and do not run on their own.

> **Note:** These commands set a 1M-token context window, except BF16 on H100, which is 256K to fit in 80GB. Lower `--max-model-len` if you are memory-constrained or want more KV-cache headroom at high concurrency.

### 1x H100

#### NVFP4

**Base (max throughput)**

For maximum throughput on this configuration, no speculative decoding is the better choice. The Mamba cache runs in FP16 because of the memory budget. Validated context is 1M tokens - lower `--max-model-len` for more KV-cache headroom at high concurrency.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --max-num-seqs 256 \
  --max-model-len 1048576 \
  --max-num-batched-tokens 16384 \
  --enable-prefix-caching \
  --async-scheduling \
  --mamba-backend flashinfer \
  --moe-backend humming \
  --linear-backend humming \
  --mamba-ssu-algorithm horizontal \
  --mamba-cache-mode align \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --reasoning-parser nemotron_v3 \
  --tool-call-parser qwen3_coder \
  --enable-auto-tool-choice
```

**Interactive, with a speculator**

The primary path above, under **Start server**, is this hardware with DSpark, tuned for interactive use at concurrency of 128 or below. To try MTP or DFlash instead, replace that command's `--speculative_config.*` flags with one of the blocks below.

**Add MTP**

```shell
--speculative_config.method mtp \
--speculative_config.num_speculative_tokens 3
```

**Add DFlash**

```shell
--speculative_config.method dflash \
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative_config.num_speculative_tokens 3
```

#### BF16

**Base**

Validated context is 256K tokens, sized to fit one 80GB H100 in BF16. The Mamba cache runs in FP16 for the same reason.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --served-model-name nemotron-3.5-lightning \
  --max-num-seqs 128 \
  --max-model-len 262144 \
  --enable-prefix-caching \
  --async-scheduling \
  --mamba-backend flashinfer \
  --mamba-ssm-cache-dtype float16 \
  --enable-mamba-cache-stochastic-rounding \
  --mamba-cache-philox-rounds 5 \
  --reasoning-parser nemotron_v3 \
  --tool-call-parser qwen3_coder \
  --enable-auto-tool-choice
```

> **Note:** BF16 weights take roughly 60GB of an 80GB H100, leaving little room for a draft model, so no speculator flags are listed here. Benchmark before assuming a speculator wins.

### 1x GB200

#### NVFP4

**Base**

Validated context is 1M tokens.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --max-model-len 1048576 \
  --max-num-batched-tokens 10240 \
  --no-enable-prefix-caching \
  --async-scheduling \
  --mamba-backend flashinfer \
  --reasoning-parser nemotron_v3 \
  --tool-call-parser qwen3_coder \
  --enable-auto-tool-choice
```

**Add MTP**

On Blackwell the MTP draft runs its MoE through Triton.

```shell
--speculative_config.method mtp \
--speculative_config.num_speculative_tokens 3 \
--speculative_config.moe_backend triton
```

**Add DFlash**

```shell
--speculative_config.method dflash \
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative_config.num_speculative_tokens 3 \
--speculative_config.attention_backend FLASHINFER
```

**Add DSpark**

```shell
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative_config.num_speculative_tokens 3
```

#### BF16

**Base**

Validated context is 1M tokens. If you are memory-constrained, or want more KV-cache headroom at high concurrency, lower `--max-model-len` and drop `VLLM_ALLOW_LONG_MAX_MODEL_LEN=1`.

```shell
VLLM_ALLOW_LONG_MAX_MODEL_LEN=1 vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --served-model-name nemotron-3.5-lightning \
  --max-num-seqs 128 \
  --max-model-len 1048576 \
  --max-num-batched-tokens 10240 \
  --no-enable-prefix-caching \
  --async-scheduling \
  --mamba-backend flashinfer \
  --reasoning-parser nemotron_v3 \
  --tool-call-parser qwen3_coder \
  --enable-auto-tool-choice
```

**Add MTP**

On Blackwell the MTP draft runs its MoE through Triton.

```shell
--speculative_config.method mtp \
--speculative_config.num_speculative_tokens 3 \
--speculative_config.moe_backend triton
```

**Add DFlash**

```shell
--speculative_config.method dflash \
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative_config.num_speculative_tokens 3 \
--speculative_config.attention_backend FLASHINFER
```

**Add DSpark**

```shell
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative_config.num_speculative_tokens 3
```

### 1x DGX Spark

#### NVFP4

**Base**

Validated context is 1M tokens.

```shell
vllm serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --served-model-name nemotron-3.5-lightning \
  --max-model-len 1048576 \
  --moe-backend marlin \
  --kv-cache-dtype fp8 \
  --enable-prefix-caching \
  --gpu-memory-utilization 0.91 \
  --mamba-backend flashinfer \
  --mamba-cache-mode align \
  --reasoning-parser nemotron_v3 \
  --tool-call-parser qwen3_coder \
  --enable-auto-tool-choice
```

**Add MTP**

```shell
--speculative_config.method mtp \
--speculative_config.num_speculative_tokens 1 \
--speculative_config.moe_backend flashinfer_cutlass
```

**Add DFlash**

```shell
--speculative_config.method dflash \
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DFlash \
--speculative_config.num_speculative_tokens 3 \
--speculative_config.attention_backend FLASHINFER
```

**Add DSpark**

The recommended recipe for DGX Spark, and what the command above was tuned alongside.

```shell
--speculative_config.model nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4-DSpark \
--speculative_config.num_speculative_tokens 3
```

The MTP and DFlash flags above were tuned at a 50K-64K context window rather than 1M, so lower `--max-model-len` on the base command if either runs out of memory.